# 🏗️ CEM4644 · MP3 — Object detection for construction
## Workshop (in class): *Workers and PPE on construction sites*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 90 minutes)**
1. Look at labelled photos, count objects yourself, and label a few photos by hand.
2. Run a trained detector, read its confidence, and move the confidence threshold.
3. Measure how many objects it finds and how many false alarms it raises; look at its mistakes.
4. Try to break it: tricky photos, edited photos, a general-purpose detector, photos from another world, your own photos.
5. Turn boxes into a site decision: a PPE compliance or equipment-count dashboard.
6. Train your own detector and compare it with the course model.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. Detection works on CPU too, but the training step in Part 6 is much faster with a GPU.

**Dataset:** Workers and PPE on construction sites. Sources and licences are listed at the bottom.

In [ ]:
#@title ▶ Step 0 · Run me first (about 2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the photos and the course model and installs the detection library.
#@markdown If Colab asks whether to run a notebook that was not authored by Google, choose *Run anyway*.
import os, sys, subprocess
if not os.path.isdir("CEM4644/mp3_object_detection"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git"], check=True)
sys.path.insert(0, os.path.abspath("CEM4644/mp3_object_detection"))
from aec_det import lab
lab.setup(dataset="construction_safety")


### ☕ While Step 0 runs: try a general-purpose detector in your browser

Open **https://www.ultralytics.com/yolo** in a new tab (no account needed). Use the *webcam* option, or upload a photo, and move the **confidence** slider.

- Point the camera at the room: it will find *person*, *chair*, *laptop*, *bottle*... These are the 80 everyday classes it was trained on.
- Now think about a construction site: it has no idea what a *helmet*, a *vest* or an *excavator* is. That is the gap this notebook closes with **fine-tuning** (Part 3, Step 3c).
- Notice how the number of boxes changes when you move the confidence slider. You will measure exactly that in Part 2.

## Part 1 · Meet the data

Classification (MP2) answers *what is in this photo?* **Object detection** answers *what is where?*: for every object it returns a **box**, a **class** and a **confidence**. Training data therefore needs a box drawn by a person around every object of interest. Every worker is boxed as *person*, with a second box for the head (*helmet* or *NO helmet*) and a third for the torso (*vest* or *NO vest*).

Boxes drawn with a **dashed** line are labels made by people; boxes with a **solid** line (later) are the model's detections.

In [ ]:
#@title ▶ Step 1a · Browse the labelled photos { display-mode: "form" }
#@markdown Pick a class to see photos that contain it. Untick *with_boxes* to see the raw photos. Click ▶ again for a new selection.
category = "all" #@param ["all", "helmet", "NO helmet", "NO vest", "person", "vest"]
how_many = 6 #@param [3, 6, 9] {type:"raw"}
with_boxes = True #@param {type:"boolean"}
lab.show_gallery(category, how_many, with_boxes)


In [ ]:
#@title ▶ Step 1b · Can *you* count them? { display-mode: "form" }
#@markdown A photo appears without boxes. Question: **How many workers are NOT wearing a helmet?** Click a number, then *Next photo*.
rounds = 6 #@param {type:"slider", min:3, max:12, step:1}
lab.count_game(rounds)


> ### 📝 Report question 1
> What was your score in the counting game? Which photos were hard for **you** (small or distant objects, objects partly hidden, unclear cases), and would you have drawn the boxes the same way as the labellers?

In [ ]:
#@title ▶ Step 1c · Label a few photos yourself { display-mode: "form" }
#@markdown Every box in the dataset was drawn by a person. Now it is your turn: draw a box around **every** object of every class, pick the class under the photo, then *Submit*. After each photo you see how your boxes compare with the dataset labels, and at the end how long the whole training set would take at your speed.
#@markdown If the drawing tool does not appear, run Step 0 again and then this cell.
how_many = 3 #@param [2, 3, 5] {type:"raw"}
lab.label_yourself(how_many)


> ### 📝 Report question 2
> Labelling: how long did you need per photo, and how many of your boxes agreed with the dataset labels? Which objects or classes were hard to decide? At your speed, how many hours would the whole training set take, and what does that mean for anyone who wants a detector for their own site?

## Part 2 · Run a trained detector

The **course model** is a YOLO detector that was fine-tuned on 897 labelled photos. For every box it proposes it also gives a **confidence** (0–100 %). A **confidence threshold** decides which boxes you keep: everything below it is thrown away.

To score a detector we compare its boxes with the labelled boxes on unseen test photos. A detection is **correct** when it has the right class and overlaps the true box by at least half. From that we count, per class:
- **recall** = share of the true objects that were found (100 % = nothing missed);
- **precision** = share of the detections that were right (100 % = no false alarms);
- **mAP50** = one overall quality score (0–100) that summarises precision and recall over all thresholds.

In [ ]:
#@title ▶ Step 2a · Detect, one photo at a time { display-mode: "form" }
#@markdown Click *🎲 Another photo* and move the *confidence ≥* slider. Watch boxes appear and disappear. Tick the checkbox to overlay the true boxes (dashed).
lab.pick_and_detect()


In [ ]:
#@title ▶ Step 2b · Score it on all the unseen test photos { display-mode: "form" }
#@markdown Per class: how many true objects were found, how many were missed, how many detections were false alarms.
how_many = "all" #@param ["all", "100"]
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.evaluate(how_many, threshold)


In [ ]:
#@title ▶ Step 2c · Move the threshold { display-mode: "form" }
#@markdown The table and the two curves update as you move the slider. Recall and precision pull in opposite directions.
lab.threshold_explorer()


In [ ]:
#@title ▶ Step 2d · Look at the mistakes { display-mode: "form" }
#@markdown Green = correct, orange = false alarm, red dashed = missed object. Choose the kind of mistake and the class; the worst photos come first.
lab.error_explorer()


> ### 📝 Report question 3
> Which class does the model find most reliably and which one does it miss most often (give the recall numbers)? Look at the missed objects in Step 2d: what do they have in common (size, distance, lighting, overlap, rarity in the training data)?

> ### 📝 Report question 4
> Set the threshold to 0.2 and to 0.8. What happens to the number of missed objects and to the number of false alarms? Which threshold would you choose for an automatic site alarm, and which for a weekly report? Explain the difference.

## Part 3 · Where does it break?

A detector only knows the kind of photos it was trained on. Let's find its limits.

In [ ]:
#@title ▶ Step 3a · Tricky photos { display-mode: "form" }
#@markdown *hard_real*: test photos with the most mistakes · *synthetic*: real photos we edited (far away, dark, blurred, rotated) · *other_domain*: photos from a different dataset · *out_of_scope*: not a site at all.
#@markdown The caption gives the labelled truth where there is one; the line below it is what the model found.
group = "all" #@param ["all", "hard_real", "crowded", "synthetic", "other_domain", "out_of_scope"]
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.tricky(group, threshold)


In [ ]:
#@title ▶ Step 3b · Break it yourself { display-mode: "form" }
#@markdown Move the sliders: rotate, zoom, *move away* (everything becomes small), blur, darken, shadow, noise. The detector re-runs after every change. Find the smallest change that makes it lose an object.
lab.playground()


In [ ]:
#@title ▶ Step 3c · A general-purpose detector vs. the course model { display-mode: "form" }
#@markdown Left: YOLO exactly as downloaded, trained on 80 everyday classes (person, car, truck...). Right: the same network after fine-tuning on our photos. Same photos, same threshold.
how_many = 3 #@param {type:"slider", min:1, max:6, step:1}
threshold = 0.4 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.compare_pretrained(how_many, threshold)


In [ ]:
#@title ▶ Step 3d · Photos from a different world { display-mode: "form" }
#@markdown Photos from **Construction machinery: excavators, dump trucks, wheel loaders** are given to the course model and to a model trained on that other dataset. Each model can only answer with its own classes.
how_many = 3 #@param {type:"slider", min:1, max:6, step:1}
threshold = 0.4 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.domain_shift(how_many, threshold)


In [ ]:
#@title ▶ Step 3e · Your own photo { display-mode: "form" }
#@markdown A small app appears with two tabs. *Photo*: upload a photo of people wearing (or not wearing) helmets and high-visibility vests, or any photo at all. *Live camera*: open the public link on your phone, allow the camera and point it at the room or the site; boxes update about once or twice a second. The confidence slider works in both tabs.
#@markdown Test at least 1 photo(s) of your own and take screenshots for your report.
lab.upload_app()


> ### 📝 Report question 5
> List three photos (tricky gallery, sliders, or your own) where the detector missed something or invented something. For each, say what it found, what it should have found, and what you think confused it.

> ### 📝 Report question 6
> In Step 3c, what does the general-purpose YOLO see in a site photo, and what does the fine-tuned course model add? In Step 3d, what happens when a model gets photos from the other dataset? What does this tell you about buying an 'AI camera' for your own site?

## Part 4 · From boxes to decisions

Boxes alone are not a decision. A safety officer wants **numbers**: how many heads without a helmet, which photos need a follow-up. The dashboard below counts the model's boxes over all test photos and compares them with the labelled truth.

In [ ]:
#@title ▶ Step 4a · Dashboard { display-mode: "form" }
#@markdown Run it several times with different thresholds and compare the numbers.
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.dashboard(threshold)


> ### 📝 Report question 7
> From the dashboard: what compliance rate does the AI report and what do the labels say? How many photos would be flagged wrongly, and how many flags would be missed? Run it at threshold 0.3 and 0.7 and explain which one you would use for (a) an instant alarm on site and (b) a monthly safety statistic.

## Part 5 · Train your own detector

**Training** shows the network labelled photos, lets it predict boxes, and nudges it every time it is wrong. One pass over the training photos is one **epoch**. The course model started from a network **pretrained** on 120,000 everyday photos (COCO) and was then **fine-tuned** on our site photos. You can start from that pretrained network or from a **random** one.

A detector needs many more training steps than the classifier in MP2, so the runs here are longer. Suggested ladder (on a GPU each run takes one to two minutes): 60 photos · 10 passes → 120 · 15 → all · 12 → the best setting with a *random* start. Without a GPU stay with 60 photos and 10 passes (about five minutes).

In [ ]:
#@title ▶ Step 5a · Train { display-mode: "form" }
#@markdown Choose the settings, name the run, click ▶. The quality score (mAP50) on the validation photos is printed after every pass; the final score on the unseen test photos is printed at the end.
training_photos = "120" #@param ["60", "120", "all"]
passes = 10 #@param {type:"slider", min:3, max:15, step:1}
start = "pretrained" #@param ["pretrained", "random"]
run_name = "run 1" #@param {type:"string"}
n = 10**9 if training_photos == "all" else int(training_photos)
lab.train_my_model(n, passes, start, run_name)


In [ ]:
#@title ▶ Step 5b · Leaderboard { display-mode: "form" }
#@markdown All your runs, best first. Copy this table into your report.
lab.leaderboard()


In [ ]:
#@title ▶ Step 5c · Your model vs. the course model on the tricky photos { display-mode: "form" }
group = "all" #@param ["all", "hard_real", "crowded", "synthetic", "other_domain", "out_of_scope"]
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.compare_my_model(group, threshold)


> ### 📝 Report question 8
> Copy your leaderboard. How did the quality score change with more photos and more passes? What happened with a random start? Why does a detector need far more training than the classifier in MP2 to reach a useful score?

> ### 📝 Report question 9
> Imagine this detector running on a site camera. Where would you place the camera, what would you do with each alarm, and what could go wrong (technically and for the people being filmed)? What data would you need to collect to make it work on your own site?

## Wrap-up

In [ ]:
#@title ▶ Step 6 · Numbers for your report { display-mode: "form" }
lab.report_summary()


### Data and model sources
- **Workers and PPE on construction sites** — Photos of construction sites with every worker boxed as 'person', plus a box for the head (helmet or NO helmet) and the torso (vest or NO vest). Source: Roboflow 100 'construction-safety' benchmark set, CC-BY-4.0.
- **Construction machinery: excavators, dump trucks, wheel loaders** (used in Step 3d) — Photos of earth-moving equipment on sites and roads, each machine boxed as excavator, dump truck or wheel loader. Source: Roboflow 100 'excavators' benchmark set, CC-BY-4.0.
- Detector: YOLO11n by Ultralytics (AGPL-3.0), pretrained on COCO, fine-tuned for this course. The `ultralytics` library is installed in Step 0.
- Out-of-scope sample images: scikit-image data (public domain / CC0).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp3_object_detection`).